# 00 — Preprocessing

Menyiapkan `station_features_<g>` dari BigQuery menjadi `df_train` / `df_val` /
`df_test` yang sudah diisi NaN dan di-scale (MinMaxScaler per stasiun), dengan
`<g>` = `GRANULARITY` (default `daily`, sama dengan notebook `01`–`05`).
NaN diisi median per jam per stasiun; untuk data daily `index.hour` selalu 0,
jadi efektifnya median keseluruhan per stasiun.

Hasilnya ditulis kembali ke BigQuery (dataset `tables`) supaya bisa dipakai
ulang oleh notebook model manapun (`01_Prophet`, `02_LSTM`, `03_GRU`,
`04_TFT`) tanpa mengulang query + preprocessing di tiap notebook, dan tanpa
bergantung pada Google Drive / disk lokal sama sekali. Logikanya identik
dengan `src/preprocessing` (`make preprocess`), jadi cukup jalankan salah satu.

Tabel yang ditulis:

- `preprocessed_data_<g>` — df_train/val/test digabung dengan kolom `split`.
- `preprocessing_metadata_<g>` — satu baris JSON berisi daftar kolom kalender,
  stasiun, fitur, dan mapping nama stasiun -> nama kolom BigQuery.
- `preprocessing_scalers_<g>` — satu baris per stasiun, `MinMaxScaler` yang
  di-pickle sebagai `BYTES`.

Input `station_features_<g>` dibuat oleh `make raw-preprocess-bq`. Jalankan
notebook ini SEKALI dulu sebelum menjalankan notebook model.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

PROJECT_ID = "thesis-506605"
DATASET    = "tables"
LOCATION   = "US"

# hourly | daily — harus sama dengan granularity notebook model (01-05: daily)
GRANULARITY = "daily"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

query = f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATASET}.station_features_{GRANULARITY}`
    ORDER BY datetime
"""
df_fe = client.query(query).to_dataframe()
df_fe = df_fe.set_index('datetime')  # restore your original index shape

# ready to feed into your model

In [ ]:
import re
import json
import pickle
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

CALENDAR_COLS = ["hour", "month", "day_of_week", "is_weekend", "is_holiday", "is_rainy_season"]
station_cols  = [c for c in df_fe.columns if c not in CALENDAR_COLS]

print(f"Loaded: {df_fe.shape} | {df_fe.index[0].date()} \u2192 {df_fe.index[-1].date()}")
print(f"Stations : {station_cols}")
print(f"NaN total: {df_fe[station_cols].isna().sum().sum():,}")

In [ ]:
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15

n = len(df_fe)
train_end_idx = int(n * TRAIN_RATIO)
val_end_idx   = int(n * (TRAIN_RATIO + VAL_RATIO))

df_train_raw = df_fe.iloc[:train_end_idx]
df_val_raw   = df_fe.iloc[train_end_idx:val_end_idx]
df_test_raw  = df_fe.iloc[val_end_idx:]

print(f"Train : {df_train_raw.index[0].date()} \u2192 {df_train_raw.index[-1].date()} ({len(df_train_raw):,} rows, {len(df_train_raw)/n:.1%})")
print(f"Val   : {df_val_raw.index[0].date()} \u2192 {df_val_raw.index[-1].date()} ({len(df_val_raw):,} rows, {len(df_val_raw)/n:.1%})")
print(f"Test  : {df_test_raw.index[0].date()} \u2192 {df_test_raw.index[-1].date()} ({len(df_test_raw):,} rows, {len(df_test_raw)/n:.1%})")

In [ ]:
# Hitung hourly median per stasiun dari training set (raw cm).
# Untuk data daily index.hour selalu 0, jadi ini jadi median keseluruhan per stasiun.
hourly_medians = {}

for station in station_cols:
    hourly_medians[station] = (
        df_train_raw[station]
        .groupby(df_train_raw.index.hour)
        .median()
    )

print("Hourly medians per stasiun (cm):")
for station, med in hourly_medians.items():
    print(f"  {station}: min={med.min():.1f} cm, max={med.max():.1f} cm")

In [ ]:
def fill_nan_hourly_median(df, hourly_medians, station_cols):
    df = df.copy()
    for station in station_cols:
        nan_mask = df[station].isna()
        if nan_mask.sum() == 0:
            continue
        df.loc[nan_mask, station] = df.loc[nan_mask].index.hour.map(
            hourly_medians[station]
        )
        print(f"  [{station}] filled {nan_mask.sum()} NaN")
    return df

df_train_raw = fill_nan_hourly_median(df_train_raw, hourly_medians, station_cols)
df_val_raw   = fill_nan_hourly_median(df_val_raw,   hourly_medians, station_cols)
df_test_raw  = fill_nan_hourly_median(df_test_raw,  hourly_medians, station_cols)

total_nan = (df_train_raw[station_cols].isna().sum().sum() +
             df_val_raw[station_cols].isna().sum().sum() +
             df_test_raw[station_cols].isna().sum().sum())
print(f"\nTotal NaN tersisa: {total_nan}")

In [ ]:
scalers = {}

for station in station_cols:
    scaler = MinMaxScaler(feature_range=(0, 1))
    valid_vals = df_train_raw[station].dropna().values.reshape(-1, 1)
    scaler.fit(valid_vals)
    scalers[station] = scaler
    print(f"  [{station}] min={scaler.data_min_[0]:.1f} cm, max={scaler.data_max_[0]:.1f} cm")

In [ ]:
def apply_scalers(df, scalers, station_cols):
    df = df.copy()
    for station in station_cols:
        mask = df[station].notna()
        df.loc[mask, station] = scalers[station].transform(
            df.loc[mask, station].values.reshape(-1, 1)
        ).flatten()
    return df

df_train = apply_scalers(df_train_raw, scalers, station_cols)
df_val   = apply_scalers(df_val_raw,   scalers, station_cols)
df_test  = apply_scalers(df_test_raw,  scalers, station_cols)

print(f"Train range: {df_train[station_cols].min().min():.4f} \u2013 {df_train[station_cols].max().max():.4f}")
print(f"Val   range: {df_val[station_cols].min().min():.4f} \u2013 {df_val[station_cols].max().max():.4f}")
print(f"Test  range: {df_test[station_cols].min().min():.4f} \u2013 {df_test[station_cols].max().max():.4f}")

## Simpan ke BigQuery

Nama stasiun (mis. "Bendung Katulampa") mengandung spasi yang tidak valid
sebagai nama kolom BigQuery, jadi disanitasi dulu (`column_map`) dan
disimpan di `preprocessing_metadata` agar notebook model bisa memetakan
baliknya.

In [ ]:
PREPROCESSED_TABLE = f"{PROJECT_ID}.{DATASET}.preprocessed_data_{GRANULARITY}"
METADATA_TABLE     = f"{PROJECT_ID}.{DATASET}.preprocessing_metadata_{GRANULARITY}"
SCALERS_TABLE      = f"{PROJECT_ID}.{DATASET}.preprocessing_scalers_{GRANULARITY}"

WRITE_DISPOSITION = "WRITE_TRUNCATE"      # atau "WRITE_APPEND"


def load_df(df, table_id, schema=None):
    job_config = bigquery.LoadJobConfig(write_disposition=WRITE_DISPOSITION)
    if schema is not None:
        job_config.schema = schema
    else:
        job_config.autodetect = True
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()   # tunggu selesai
    tbl = client.get_table(table_id)
    print(f"  {table_id}: {tbl.num_rows:,} rows")

In [ ]:
def to_bq_column(name):
    """Nama kolom BigQuery harus alfanumerik + underscore."""
    return re.sub(r"[^0-9a-zA-Z_]", "_", name)

column_map = {station: to_bq_column(station) for station in station_cols}

def make_bq_frame(df, split):
    out = df.rename(columns=column_map).reset_index()
    out["split"] = split
    return out

preprocessed_df = pd.concat([
    make_bq_frame(df_train, "train"),
    make_bq_frame(df_val,   "val"),
    make_bq_frame(df_test,  "test"),
], ignore_index=True)

load_df(preprocessed_df, PREPROCESSED_TABLE)   # schema autodetect (kolom stasiun dinamis)

In [ ]:
saved_at = pd.Timestamp.now(tz="UTC").tz_localize(None)

metadata = {
    "CALENDAR_COLS": CALENDAR_COLS,
    "station_cols": station_cols,
    "feature_cols": station_cols + CALENDAR_COLS,
    "column_map": column_map,   # station -> kolom BigQuery
}
metadata_schema = [
    bigquery.SchemaField("payload", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("saved_at", "TIMESTAMP", mode="REQUIRED"),
]
metadata_df = pd.DataFrame([{
    "payload": json.dumps(metadata),
    "saved_at": saved_at,
}])
load_df(metadata_df, METADATA_TABLE, metadata_schema)

scalers_schema = [
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("scaler_blob", "BYTES", mode="REQUIRED"),
    bigquery.SchemaField("saved_at", "TIMESTAMP", mode="REQUIRED"),
]
scalers_df = pd.DataFrame([{
    "station": station,
    "scaler_blob": pickle.dumps(scaler),
    "saved_at": saved_at,
} for station, scaler in scalers.items()])
load_df(scalers_df, SCALERS_TABLE, scalers_schema)

print(f"Saved ({GRANULARITY}): preprocessed_data, preprocessing_metadata, preprocessing_scalers -> BigQuery")